In [1]:
!nvidia-smi

Thu Jul  2 12:53:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.153.02             Driver Version: 570.153.02     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200 NVL                Off |   00000000:03:00.0 Off |                    0 |
| N/A   40C    P0            100W /  600W |    2609MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
## Imports
import numpy as np
import torch
import json
import os
from tabulate import tabulate
from PIL import Image
from torch.nn import functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision.datasets import ImageNet, CIFAR10, CIFAR100, ImageFolder
from utils.misc.misc import accuracy, accuracy_correct
from utils.scripts.algorithms_text_explanations import svd_data_approx
from utils.scripts.algorithms_text_explanations_funcs import *
from utils.models.factory import create_model_and_transforms, get_tokenizer
from utils.models.prs_hook import hook_prs_logger
from utils.models.prs_hook_text import hook_prs_logger_text
from utils.misc.visualization import visualization_preprocess
from utils.datasets_constants.imagenet_classes import imagenet_classes
from utils.datasets_constants.cifar_10_classes import cifar_10_classes
from utils.datasets_constants.cub_classes import cub_classes, waterbird_classes
from utils.datasets.dataset_helpers import dataset_subset
from utils.datasets.binary_waterbirds import BinaryWaterbirds


/home/virgilio/lib/minconda3/envs/MT/lib/python3.10/site-packages/torch/cuda/__init__.py:51: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
## Parameters
device = 'cpu'
model_name = 'ViT-B-32'        # 'ViT-H-14', 'ViT-L-14', 'ViT-B-16', 'ViT-B-32'
seed = 0
num_last_layers_ = 6            # how many of the LAST layers get a full PC decomposition, for BOTH encoders
algorithm = "svd_data_approx"

# Dataset the vision is run on (both for the activations / visualization)
dataset_image_name = "imagenet"      
subset_dim = 10 
tot_samples_per_class = 50 # nr samples per class
path = './datasets/'

# Dataset the text encoder is run to produce activations
dataset = "imagenet_descriptions_personal" 
native_per_class = 10           # sentences per class in "dataset"
sentences_per_class = 1         # keep the LAST k per class; 1 -> only the class-name sentence

# Shared: labels used to text-label PCs of BOTH encoders
dataset_text_name = "top_1500_nouns_5_sentences_imagenet_clean"

cache_dir = "../cache"

if model_name == "ViT-H-14":
    pretrained = "laion2B-s32B-b79K"; precision = "fp32"
elif model_name == "ViT-L-14":
    pretrained = "laion2B-s32B-b82K"; precision = "fp32"
elif model_name == "ViT-B-16":
    pretrained = "laion2B-s34B-b88K"; precision = "fp32"
elif model_name == "ViT-B-32":
    pretrained = "laion2B-s34B-b79K"; precision = "fp32"
elif model_name == "ViT-L-14-336":
    pretrained = "openai"; precision = "fp16"


In [4]:
## Loading Model — one CLIP checkpoint, both towers
model, _, preprocess = create_model_and_transforms(
    model_name, pretrained=pretrained, precision=precision, cache_dir=cache_dir)
model.to(device)
model.eval()
tokenizer = get_tokenizer(model_name)

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Number of VISION-encoder layers:", len(model.visual.transformer.resblocks))
print("Number of TEXT-encoder layers:", len(model.transformer.resblocks))


: 

: 

: 

# Run the spectral decomposition for both encoders


In [ ]:
## Run the chosen algorithm on the VISION-encoder attention heads approximating the PCs using the text descriptions
## and evaluate the accuracy of reconstruction over subset of imagenet used
command_v = (f"python -m utils.scripts.compute_text_explanations "
             f"--device {device} --model {model_name} --algorithm {algorithm} --seed {seed} "
             f"--text_per_princ_comp 20 --num_of_last_layers {num_last_layers_} "
             f"--text_descriptions {dataset_text_name} --dataset {dataset_image_name}")
!{command_v}


In [ ]:
## Run the chosen algorithm on the TEXT-encoder heads (and reconstruct using probe_modality)
## and evaluate zero-shot accuracy over the given images
command_t = (f"python -m utils.scripts.compute_text_explanations_text "
             f"--device {device} --model {model_name} --algorithm {algorithm} --seed {seed} "
             f"--text_per_princ_comp 20 --num_of_last_layers {num_last_layers_} "
             f"--dataset {dataset} --text_descriptions {dataset_text_name} --probe_modality text") 
!{command_t}


# Load precomputed data for both encoders


In [4]:
# VISION encoder: components, classifier, labels
attention_dataset_v = f"output_dir/{dataset_image_name}_completeness_{dataset_text_name}_{model_name}_algo_{algorithm}_seed_{seed}.jsonl"

attns_v = torch.tensor(np.load(f"output_dir/{dataset_image_name}_attn_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device, dtype=torch.float32)  # [N, l, h, d]
mlps_v  = torch.tensor(np.load(f"output_dir/{dataset_image_name}_mlp_{model_name}_seed_{seed}.npy",  mmap_mode="r")).to(device, dtype=torch.float32)  # [N, l+1, d]
classifier_ = torch.tensor(np.load(f"output_dir/{dataset_image_name}_classifier_{model_name}.npy", mmap_mode="r")).to(device, dtype=torch.float32)   # [d, C], class-prompt classifier
labels_v = torch.tensor(np.load(f"output_dir/{dataset_image_name}_labels_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device).long()            # [N], class per image

# Shared probes: label (texts) + image probes, used to interpret heads/PCs of BOTH encoders
final_embeddings_images = torch.tensor(np.load(f"output_dir/{dataset_image_name}_embeddings_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device, dtype=torch.float32)
final_embeddings_texts  = torch.tensor(np.load(f"output_dir/{dataset_text_name}_{model_name}.npy", mmap_mode="r")).to(device, dtype=torch.float32)
with open(f"utils/text_descriptions/{dataset_text_name}.txt", "r") as f:
    texts_str = np.array([i.replace("\n", "") for i in f.readlines()])

# Derived quantities (mirrors playground.ipynb)
no_heads_attentions_v = attns_v.sum(axis=2)                # [N, l, d], summed over heads
nr_layers_v = attns_v.shape[1]
nr_heads_v  = attns_v.shape[2]
last_v = nr_layers_v - num_last_layers_                    # first layer index that got a PC decomposition
current_mean_ablation_per_head_sum_v = torch.mean(no_heads_attentions_v[:, :last_v + 1], axis=0).sum(0)

ds_ = ImageNet(root=path + "imagenet/", split="val", transform=visualization_preprocess)
classes_ = imagenet_classes
ds_vis_ = dataset_subset(ds_, samples_per_class=subset_dim, tot_samples_per_class=tot_samples_per_class, seed=seed)

data_v_full = get_data(attention_dataset_v, -1)                       # all PCs, incl. head=-1 summary line
data_v = get_data(attention_dataset_v, -1, skip_final=True)           # all PCs, per-head only
mean_rank_v = sum(e["rank"] for e in data_v) / len(data_v)
print("VISION  attns", tuple(attns_v.shape), "mlps", tuple(mlps_v.shape), "| mean head rank:", round(mean_rank_v, 2))


: 

: 

In [ ]:
# TEXT encoder: components, labels
attention_dataset_t = f"output_dir/{dataset}_completeness_text_{dataset_text_name}_{model_name}_algo_{algorithm}_seed_{seed}.jsonl"

attns_t = torch.tensor(np.load(f"output_dir/{dataset}_attn_text_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device, dtype=torch.float32)  # [N, l, h, d]
mlps_t  = torch.tensor(np.load(f"output_dir/{dataset}_mlp_text_{model_name}_seed_{seed}.npy",  mmap_mode="r")).to(device, dtype=torch.float32)  # [N, l+1, d]

# Decomposed-corpus texts, aligned with attns_t rows (same last-k-per-class subsampling as prepare)
with open(f"utils/text_descriptions/{dataset}.txt", "r") as f:
    _all = [i.replace("\n", "") for i in f.readlines()]
corpus_texts = np.array([l for c in range(len(_all) // native_per_class)
                         for l in _all[(c + 1) * native_per_class - sentences_per_class:(c + 1) * native_per_class]])

# Class labels (row -> ImageNet class) and per-image labels, for text-encoder accuracy
labels_t = torch.tensor(np.load(f"output_dir/{dataset}_labels_text_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device).long()
image_labels_t = torch.tensor(np.load(f"output_dir/{dataset_image_name}_labels_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device).long()

# Exact decomposition target: M_text = sum_l sum_h attn + sum_l mlp  (EOS, projected)
final_embeddings_texts_decomp = attns_t.sum(axis=(1, 2)) + mlps_t.sum(axis=1)  # [N, d]

no_heads_attentions_t = attns_t.sum(axis=2)      # [N, l, d]
nr_layers_t = attns_t.shape[1]
nr_heads_t  = attns_t.shape[2]
last_t = nr_layers_t - num_last_layers_
current_mean_ablation_per_head_sum_t = torch.mean(no_heads_attentions_t[:, :last_t + 1], axis=0).sum(0)

data_t_full = get_data(attention_dataset_t, -1)
data_t = get_data(attention_dataset_t, -1, skip_final=True)
mean_rank_t = sum(e["rank"] for e in data_t) / len(data_t)
print("TEXT    attns", tuple(attns_t.shape), "mlps", tuple(mlps_t.shape), "| mean head rank:", round(mean_rank_t, 2))


# Top principal-component text interpretation for each head (vision + text)


In [ ]:
print("=== VISION-encoder heads ===")
print_data(data_v_full, min_princ_comp=4)


In [ ]:
print("=== TEXT-encoder heads ===")
print_data(data_t_full, min_princ_comp=4)


# Strongest principal components across each encoder


In [ ]:
top_k = 10
print("=== VISION: strongest PCs ===")
print_data(top_data(sort_data_by(data_v, "strength_abs", descending=True), top_k=top_k))


In [ ]:
print("=== TEXT: strongest PCs ===")
print_data(top_data(sort_data_by(data_t, "strength_abs", descending=True), top_k=top_k))


# Characterize one component bidirectionally (nearest images AND nearest texts)
Specify a `(layer, head, princ_comp)` for each encoder below; `visualize_principal_component`
scores that PC direction against both the image probes and the text probes.


In [ ]:
# VISION component to inspect
layer_v, head_v, princ_comp_v = 11, 2, 0
# TEXT component to inspect
layer_t, head_t, princ_comp_t = 10, 6, 0

nr_top_imgs = 20   # Number of top elements
nr_worst_imgs = 20  # Number of worst elements
nr_cont_imgs = 0    # Length of continuous elements


In [ ]:
print(f"VISION -- layer {layer_v}, head {head_v}, PC {princ_comp_v}")
visualize_principal_component(layer_v, head_v, princ_comp_v, nr_top_imgs, nr_worst_imgs, nr_cont_imgs,
    attention_dataset_v, final_embeddings_images, final_embeddings_texts, seed, path, texts_str,
    dataset=dataset_image_name, samples_per_class=subset_dim, tot_samples_per_class=tot_samples_per_class)


In [ ]:
print(f"TEXT -- layer {layer_t}, head {head_t}, PC {princ_comp_t}")
visualize_principal_component(layer_t, head_t, princ_comp_t, nr_top_imgs, nr_worst_imgs, nr_cont_imgs,
    attention_dataset_t, final_embeddings_images, final_embeddings_texts, seed, path, texts_str,
    dataset=dataset_image_name, samples_per_class=subset_dim, tot_samples_per_class=tot_samples_per_class)


# Singular-value (variance) profiles of the components for both encoders


In [ ]:
print("VISION:")
plot_pc_sv_across_heads_and_layers(data_v_full, nr_layers_v - 1, nr_layer=num_last_layers_)


In [ ]:
print("TEXT:")
plot_pc_sv_across_heads_and_layers(data_t_full, nr_layers_t - 1, nr_layer=num_last_layers_)


In [ ]:
# Single-head singular-value curve for the components picked above
plot_pc_sv(data_v_full, layer_v, head_v)


In [ ]:
plot_pc_sv(data_t_full, layer_t, head_t)


# Per-PC cosine similarity to images vs. texts, across the last layers, for both models


In [ ]:
def visualize_pc_cosine_similarity(
    data,
    final_embeddings_images: torch.Tensor,
    final_embeddings_texts: torch.Tensor,
    base_layer: int,
    model_name: str = "ViT-B-32",
    max_pc: int = 10,
    top_scores: int = 5,
    n_permutations: int = 10
):
    # ---- sorted ascending layers ----
    layers = sorted(base_layer - i for i in range(6))

    # ---- normalize embeddings ----
    texts_norm  = final_embeddings_texts  / final_embeddings_texts.norm(dim=-1, keepdim=True)
    images_norm = final_embeddings_images / final_embeddings_images.norm(dim=-1, keepdim=True)

    # ---- allocate arrays ----
    cos_mean_text = np.zeros((max_pc, len(layers)))
    cos_std_text  = np.zeros_like(cos_mean_text)
    cos_mean_img  = np.zeros_like(cos_mean_text)
    cos_std_img   = np.zeros_like(cos_mean_text)

    # ---- fill per-PC, per-layer statistics ----
    for j, layer in enumerate(layers):
        for i in range(max_pc):
            e = [e for e in data if e['layer'] == layer and e['princ_comp'] == i][0]
            vh     = torch.tensor(e['vh'])
            pc_dir = vh[i] / vh[i].norm()

            txt_cos = torch.topk(torch.abs(texts_norm @ pc_dir), top_scores).values
            img_cos = torch.topk(torch.abs(images_norm @ pc_dir), top_scores).values

            cos_mean_text[i, j] = txt_cos.mean().item()
            cos_std_text[i, j]  = txt_cos.std().item()
            cos_mean_img[i, j]  = img_cos.mean().item()
            cos_std_img[i, j]   = img_cos.std().item()

    # ---- build annotation strings "mean±std" ----
    ann_text = np.array([[f"{m:.2f}±{s:.2f}" for m, s in zip(row_m, row_s)]
                         for row_m, row_s in zip(cos_mean_text, cos_std_text)])
    ann_img = np.array([[f"{m:.2f}±{s:.2f}" for m, s in zip(row_m, row_s)]
                        for row_m, row_s in zip(cos_mean_img, cos_std_img)])

    # ---- determine shared color scale ----
    vmin = min(cos_mean_text.min(), cos_mean_img.min())
    vmax = max(cos_mean_text.max(), cos_mean_img.max())

    # ---- plot heatmaps ----
    fig, axes = plt.subplots(
        1, 2, figsize=(15, max_pc * 0.6 + 3),
        gridspec_kw={'width_ratios': [1, 1], 'wspace': 0.3}
    )
    fig.suptitle(f"Model: {model_name} — PC Cosine Similarities", fontsize=18, y=0.99)

    sns.heatmap(cos_mean_text, annot=ann_text, fmt="", cmap="viridis", vmin=vmin, vmax=vmax, cbar=False,
                ax=axes[0], yticklabels=[f"PC {i}" for i in range(max_pc)], xticklabels=[f"Layer {l}" for l in layers])
    axes[0].set_title("Top Text Cosine Similarity", fontsize=14)

    sns.heatmap(cos_mean_img, annot=ann_img, fmt="", cmap="viridis", vmin=vmin, vmax=vmax, cbar=False,
                ax=axes[1], yticklabels=[f"PC {i}" for i in range(max_pc)], xticklabels=[f"Layer {l}" for l in layers])
    axes[1].set_title("Top Image Cosine Similarity", fontsize=14)

    cbar_ax = fig.add_axes([0.92, 0.3, 0.02, 0.4])
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
    sm.set_array([])
    fig.colorbar(sm, cax=cbar_ax, label="Cosine Similarity")

    # ---- compute summary stats ----
    pairwise     = (texts_norm @ images_norm.T).cpu().numpy()
    global_mean  = pairwise.mean()
    global_std   = pairwise.std()

    n_txt, n_img = texts_norm.size(0), images_norm.size(0)
    pairs        = min(n_txt, n_img)
    txt_txt, img_img, txt_img = [], [], []

    for _ in range(n_permutations):
        p_t = torch.randperm(n_txt)
        txt_txt.append(((texts_norm * texts_norm[p_t]).sum(dim=1)).mean().item())

        p_i = torch.randperm(n_img)
        img_img.append(((images_norm * images_norm[p_i]).sum(dim=1)).mean().item())

        idx_t = torch.randperm(n_txt)[:pairs]
        idx_i = torch.randperm(n_img)[:pairs]
        txt_img.append(((texts_norm[idx_t] * images_norm[idx_i]).sum(dim=1)).mean().item())

    tt_mean, tt_std = np.mean(txt_txt),  np.std(txt_txt)
    ii_mean, ii_std = np.mean(img_img),  np.std(img_img)

    summary = (
        f"Text–Image: {global_mean:.2f}±{global_std:.2f}    "
        f"Text–Text (Random): {tt_mean:.2f}±{tt_std:.3f}\n"
        f"Image–Image (Random): {ii_mean:.2f}±{ii_std:.3f}    "
    )
    plt.tight_layout(rect=[0, 0.08, 1, 0.95])
    fig.text(0.5, 0, summary, ha='center', va='center', fontsize=12,
              bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="gray"))
    plt.show()


In [ ]:
visualize_pc_cosine_similarity(data_v_full, final_embeddings_images, final_embeddings_texts,
    nr_layers_v - 1, model_name=f"{model_name} (vision)", max_pc=5)


In [ ]:
visualize_pc_cosine_similarity(data_t_full, final_embeddings_images, final_embeddings_texts,
    nr_layers_t - 1, model_name=f"{model_name} (text)", max_pc=5)


# Query a topic / image and find nearest neighbours via the components of BOTH encoders
Same query, reconstructed once through the vision-encoder components and once through the
text-encoder components, so the two views are directly comparable.


In [ ]:
model.eval()
query_text = True
max_pcs_per_head = 10

with torch.no_grad():
    if query_text:
        text_query = "An image of colour black."
        topic_emb = model.encode_text(tokenizer(text_query).to(device), normalize=True).to(torch.float32)
    else:
        text_query = "woman.png"
        image = preprocess(Image.open(f'images/{text_query}'))[np.newaxis].to(device)
        if precision == "fp16":
            image = image.to(dtype=torch.float16)
        topic_emb = model.encode_image(image, attn_method='head_no_spatial', normalize=True).to(torch.float32)

mean_final_images = torch.mean(final_embeddings_images, axis=0).to(device)
mean_final_texts  = torch.mean(final_embeddings_texts,  axis=0).to(device)
mean_final = mean_final_texts if query_text else mean_final_images

topic_emb_cent = topic_emb - mean_final
topic_emb_cent_norm = topic_emb_cent / topic_emb_cent.norm(dim=-1, keepdim=True)

top_k = 30       # top-k components to keep for the greedy reconstruction
approx = 1.1     # target approximation threshold
nr_top_imgs, nr_worst_imgs, nr_cont_imgs = 20, 20, 0


### VISION-encoder components: reconstruct + bidirectional NN


In [ ]:
data_v_q = get_data(attention_dataset_v, max_pcs_per_head, skip_final=True)
[topic_emb_rec_cent_v], data_v_q = reconstruct_embeddings(
    data_v_q, [topic_emb_cent], ["text" if query_text else "image"],
    return_princ_comp=True, plot=True, means=[mean_final], device=device)
topic_emb_rec_cent_v_norm = topic_emb_rec_cent_v / topic_emb_rec_cent_v.norm(dim=-1, keepdim=True)
max_reconstr_score_v = topic_emb_rec_cent_v_norm @ topic_emb_cent_norm.T
print(f"[VISION components] max cosine similarity of the reconstruction: {max_reconstr_score_v.item():.4f}")

data_v_q = sort_data_by(data_v_q, "correlation_princ_comp_abs", descending=True)
top_k_entries_v = top_data(data_v_q, top_k)
top_k_details_v = reconstruct_top_embedding(
    top_k_entries_v, topic_emb_cent, mean_final, "text" if query_text else "image",
    max_reconstr_score_v, top_k, approx, device=device)
print(f"Querying: {text_query}")
print_data(top_k_details_v, is_corr_present=True)


In [ ]:
# Reconstruct the vision dataset's own images emphasising the query-relevant vision components
images_rec_v = reconstruct_all_embeddings_mean_ablation_pcs(
    top_k_entries_v, mlps_v, attns_v, attns_v, nr_layers_v, nr_heads_v, last_v,
    ratio=-1, ablation=True, mean_ablate_all=True).to(device)

# Restrict to the same subset ds_vis_ actually covers, so img_index stays valid for ds_vis_[img_index]
images_rec_v = (images_rec_v[ds_vis_.indices]
                if isinstance(ds_vis_, torch.utils.data.Subset) else images_rec_v)

scores_img_v = np.empty(images_rec_v.shape[0], dtype=[('score', 'f4'), ('score_vis', 'f4'), ('img_index', 'i4')])
irv = images_rec_v / images_rec_v.norm(dim=-1, keepdim=True)
scores_img_v['score']     = (images_rec_v @ topic_emb.T).squeeze().cpu().numpy()
scores_img_v['score_vis'] = (irv @ topic_emb.T).squeeze().cpu().numpy()
scores_img_v['img_index'] = np.arange(images_rec_v.shape[0])

# Text-side: raw label-bank probes scored directly (the label bank is never run through vision components)
scores_txt_v = np.empty(final_embeddings_texts.shape[0], dtype=[('score', 'f4'), ('score_vis', 'f4'), ('txt_index', 'i4')])
tnv = final_embeddings_texts / final_embeddings_texts.norm(dim=-1, keepdim=True)
scores_txt_v['score']     = (final_embeddings_texts @ topic_emb.T).squeeze().cpu().numpy()
scores_txt_v['score_vis'] = (tnv @ topic_emb.T).squeeze().cpu().numpy()
scores_txt_v['txt_index'] = np.arange(final_embeddings_texts.shape[0])

dbs_v = create_dbs(scores_img_v, scores_txt_v, nr_top_imgs, nr_worst_imgs, nr_cont_imgs,
                    text_query=text_query, max_reconstr_score=max_reconstr_score_v)
dbs_v = [db for nr, db in zip([nr_top_imgs, nr_worst_imgs, nr_cont_imgs], dbs_v) if nr > 0]
visualize_dbs(top_k_details_v, dbs_v, ds_vis_, texts_str, classes_, text_query)

### TEXT-encoder components: reconstruct + bidirectional NN


In [ ]:
data_t_q = get_data(attention_dataset_t, max_pcs_per_head, skip_final=True)
[topic_emb_rec_cent_t], data_t_q = reconstruct_embeddings(
    data_t_q, [topic_emb_cent], ["text" if query_text else "image"],
    return_princ_comp=True, plot=True, means=[mean_final], device=device)
topic_emb_rec_cent_t_norm = topic_emb_rec_cent_t / topic_emb_rec_cent_t.norm(dim=-1, keepdim=True)
max_reconstr_score_t = topic_emb_rec_cent_t_norm @ topic_emb_cent_norm.T
print(f"[TEXT components] max cosine similarity of the reconstruction: {max_reconstr_score_t.item():.4f}")

data_t_q = sort_data_by(data_t_q, "correlation_princ_comp_abs", descending=True)
top_k_entries_t = top_data(data_t_q, top_k)
top_k_details_t = reconstruct_top_embedding(
    top_k_entries_t, topic_emb_cent, mean_final, "text" if query_text else "image",
    max_reconstr_score_t, top_k, approx, device=device)
print(f"Querying: {text_query}")
print_data(top_k_details_t, is_corr_present=True)


In [ ]:
# Reconstruct the text corpus (bank-text embeddings) emphasising the query-relevant text components
texts_rec_t = reconstruct_all_embeddings_mean_ablation_pcs(
    top_k_entries_t, mlps_t, attns_t, attns_t, nr_layers_t, nr_heads_t, last_t,
    ratio=-1, ablation=True, mean_ablate_all=True).to(device)

scores_txt_t = np.empty(texts_rec_t.shape[0], dtype=[('score', 'f4'), ('score_vis', 'f4'), ('txt_index', 'i4')])
trt = texts_rec_t / texts_rec_t.norm(dim=-1, keepdim=True)
scores_txt_t['score']     = (texts_rec_t @ topic_emb.T).squeeze().cpu().numpy()
scores_txt_t['score_vis'] = (trt @ topic_emb.T).squeeze().cpu().numpy()
scores_txt_t['txt_index'] = np.arange(texts_rec_t.shape[0])

# Image-side: raw image probes scored directly, restricted to the same subset ds_vis_ covers
images_vis_t = (final_embeddings_images[ds_vis_.indices]
                if isinstance(ds_vis_, torch.utils.data.Subset) else final_embeddings_images)
scores_img_t = np.empty(images_vis_t.shape[0], dtype=[('score', 'f4'), ('score_vis', 'f4'), ('img_index', 'i4')])
fit = images_vis_t / images_vis_t.norm(dim=-1, keepdim=True)
scores_img_t['score']     = (images_vis_t @ topic_emb.T).squeeze().cpu().numpy()
scores_img_t['score_vis'] = (fit @ topic_emb.T).squeeze().cpu().numpy()
scores_img_t['img_index'] = np.arange(images_vis_t.shape[0])

dbs_t = create_dbs(scores_img_t, scores_txt_t, nr_top_imgs, nr_worst_imgs, nr_cont_imgs,
                    text_query=text_query, max_reconstr_score=max_reconstr_score_t)
dbs_t = [db for nr, db in zip([nr_top_imgs, nr_worst_imgs, nr_cont_imgs], dbs_t) if nr > 0]
visualize_dbs(top_k_details_t, dbs_t, ds_vis_, corpus_texts, classes_, text_query)


# Ablation study: mean-ablating heads and MLPs, vision and text encoders
Classification accuracy of the reconstruction as attention layers / MLP layers are progressively
mean-ablated (replaced by their dataset mean), for BOTH encoders, in three accumulation orders:
forward (layer 0 -> last), backward (last -> layer 0) and one seeded random permutation.
All layers are covered by default (not just the last `num_last_layers_`), to see which
components/positions matter most for accuracy.


In [ ]:
## Unified accuracy evaluator for both encoders
compare_against = "image"  # for the TEXT encoder only: "text" or "image" (see below)
assert compare_against in {"text", "image"}

@torch.no_grad()
def evaluate_vision(embeds, label="vision"):
    return test_accuracy(embeds @ classifier_, labels_v, label=label)[0]

@torch.no_grad()
def class_weights_text(rec):
    """Per-class prototype (mean over each class's reconstructed sentences), L2-normalized -> [C, d]."""
    C = int(labels_t.max().item()) + 1
    W = torch.stack([rec[labels_t == c].mean(0) for c in range(C)])
    return W / W.norm(dim=-1, keepdim=True)

@torch.no_grad()
def evaluate_text(embeds, label="text"):
    # "text"  -> samples = reconstructed class sentences, weights = class-prompt classifier_
    # "image" -> weights = per-class mean of the reconstruction (prototypes), samples = final_embeddings_images
    if compare_against == "text":
        return test_accuracy(embeds @ classifier_, labels_t, label=label)[0]
    imgs = final_embeddings_images / final_embeddings_images.norm(dim=-1, keepdim=True)
    pred = (imgs @ class_weights_text(embeds).T).argmax(1)
    acc = (pred == image_labels_t).float().mean().item() * 100
    print(f"For the approach {label}, the accuracy is: {acc:3f}%")
    return acc


In [ ]:
## Layer-ablation order + incremental mean-ablation accuracy curve (shared by heads and MLPs)
def ablation_order(nr_layers, kind, seed):
    """kind: 'forward' (0->last), 'backward' (last->0) or 'random' (one seeded permutation)."""
    layers = list(range(nr_layers))
    if kind == "forward":
        return layers
    if kind == "backward":
        return layers[::-1]
    if kind == "random":
        order = layers.copy()
        np.random.RandomState(seed).shuffle(order)
        return order
    raise ValueError(kind)

def mean_ablate_curve(component_sum, other_component_sum, order, evaluate_fn, label):
    """
    component_sum:       [N, L, d] per-sample per-layer contribution to progressively mean-ablate
                          (e.g. no_heads_attentions_v for heads, or mlps_v for MLPs).
    other_component_sum: [N, d] the OTHER component type's full sum, held fixed.
    order:                layer-index accumulation order (see ablation_order).
    evaluate_fn:          callable(embeds, label) -> accuracy.
    """
    L = component_sum.shape[1]
    layer_mean = component_sum.mean(0)  # [L, d]
    ablated_mask = torch.zeros(L, dtype=torch.bool)
    accs = [evaluate_fn(component_sum.sum(1) + other_component_sum, label=f"{label} (0 ablated)")]
    for i, l in enumerate(order):
        ablated_mask[l] = True
        kept = component_sum * (~ablated_mask)[None, :, None]
        replaced = layer_mean[None, :, :] * ablated_mask[None, :, None]
        embeds = (kept + replaced).sum(1) + other_component_sum
        accs.append(evaluate_fn(embeds, label=f"{label} ({i + 1} ablated)"))
    return accs

n_random_trials = 5  # number of random layer-orders to average for the 'random' curve

def mean_ablate_curve_random_trials(component_sum, other_component_sum, evaluate_fn, label, seed, n_trials=5):
    """Run mean_ablate_curve for n_trials random layer orders (seeds seed..seed+n_trials-1)
    and return (mean_accs, std_accs) across trials, each of length nr_layers + 1."""
    nr_layers = component_sum.shape[1]
    trials = []
    for t in range(n_trials):
        order = ablation_order(nr_layers, "random", seed + t)
        trials.append(mean_ablate_curve(component_sum, other_component_sum, order, evaluate_fn,
                                         label=f"{label}-random (trial {t})"))
    trials = np.array(trials)  # [n_trials, nr_layers + 1]
    return trials.mean(axis=0), trials.std(axis=0)

In [ ]:
## Classification accuracy of the reconstruction under mean-ablation, on BOTH encoders
def plot_ablation_panel(ax, component_sum, other_component_sum, evaluate_fn, label_prefix, seed, n_random_trials):
    nr_layers = component_sum.shape[1]
    x = range(nr_layers + 1)

    for kind in ["forward", "backward"]:
        order = ablation_order(nr_layers, kind, seed)
        accs = mean_ablate_curve(component_sum, other_component_sum, order, evaluate_fn, label=f"{label_prefix}-{kind}")
        ax.plot(x, accs, marker='o', label=kind)

    mean_accs, std_accs = mean_ablate_curve_random_trials(
        component_sum, other_component_sum, evaluate_fn, label_prefix, seed, n_trials=n_random_trials)
    ax.plot(x, mean_accs, marker='o', label=f"random (mean of {n_random_trials})")
    ax.fill_between(x, mean_accs - std_accs, mean_accs + std_accs, alpha=0.2)


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

plot_ablation_panel(axes[0, 0], no_heads_attentions_v, mlps_v.sum(1),
                     lambda e, label: evaluate_vision(e, label=label), "vision-heads", seed, n_random_trials)
axes[0, 0].set_title("Vision: attention heads"); axes[0, 0].set_xlabel("# layers mean-ablated"); axes[0, 0].set_ylabel("Accuracy (%)")
axes[0, 0].legend(); axes[0, 0].grid(True)

plot_ablation_panel(axes[0, 1], mlps_v, attns_v.sum(axis=(1, 2)),
                     lambda e, label: evaluate_vision(e, label=label), "vision-mlp", seed, n_random_trials)
axes[0, 1].set_title("Vision: MLP layers"); axes[0, 1].set_xlabel("# layers mean-ablated")
axes[0, 1].legend(); axes[0, 1].grid(True)

plot_ablation_panel(axes[1, 0], no_heads_attentions_t, mlps_t.sum(1),
                     evaluate_text, "text-heads", seed, n_random_trials)
axes[1, 0].set_title("Text: attention heads"); axes[1, 0].set_xlabel("# layers mean-ablated"); axes[1, 0].set_ylabel(f"Accuracy (compare_against={compare_against})")
axes[1, 0].legend(); axes[1, 0].grid(True)

plot_ablation_panel(axes[1, 1], mlps_t, attns_t.sum(axis=(1, 2)),
                     evaluate_text, "text-mlp", seed, n_random_trials)
axes[1, 1].set_title("Text: MLP layers"); axes[1, 1].set_xlabel("# layers mean-ablated")
axes[1, 1].legend(); axes[1, 1].grid(True)

fig.suptitle(f"Accuracy vs. mean-ablation ({model_name}) — forward / backward / random (mean ± std over {n_random_trials} trials)")
plt.tight_layout()
plt.show()

# Cross-modal component similarity: vision vs. text components
Specify a sentence (or a list of sentences) and an image (or a list of images), run both
encoders on them, and compare every vision component against every text component via a
dot-product / cosine heatmap. Two granularities are computed:
- **head/MLP-layer level** -- component = raw mean contribution vector of a `(layer, head)`
  attention head or of a whole MLP layer (no PC decomposition; MLPs are never decomposed
  into PCs elsewhere in this codebase either).
- **PC level** -- component = an individual principal-component direction `vh[i]`. Attention
  heads reuse the PCs already computed above; MLP layers get an on-the-fly decomposition via
  the same `svd_data_approx` used for heads, applied directly to the MLP layer's activation
  matrix (restricted to the last `num_last_layers_` layers, matching the heads' scope).

Run once for the specified query (averaged over the given sentence(s)/image(s)) and once for
the whole dataset (averaging the already-loaded `attns_/mlps_` tensors: the full ImageNet val
set for vision, the full `imagenet_descriptions_personal` text corpus for text).


In [ ]:
## Specify the query
query_sentences = ["A photo of a golden retriever running on the grass."]
query_images = ["images/woman.png"]     # paths under the repo root; use a list for multiple images

heatmap_metric = "cosine"        # "dot" or "cosine"
top_k_pairs = 10               # how many most-similar (vision, text) component pairs to print
max_pcs_per_component = 5      # PCs kept per head/MLP-layer for the PC-level heatmaps


In [ ]:
## Run the VISION encoder on the query image(s) and average the per-component contributions
prs_v_query = hook_prs_logger(model, device, spatial=False)
q_attns_v_list, q_mlps_v_list = [], []
with torch.no_grad():
    for img_path in query_images:
        prs_v_query.reinit()
        image = preprocess(Image.open(img_path).convert("RGB"))[np.newaxis].to(device)
        if precision == "fp16":
            image = image.to(dtype=torch.float16)
        representation = model.encode_image(image, attn_method="head_no_spatial", normalize=False)
        attentions, mlps = prs_v_query.finalize(representation)
        q_attns_v_list.append(attentions[0].float())   # [l, h, d]
        q_mlps_v_list.append(mlps[0].float())           # [l+1, d]
q_attn_v = torch.stack(q_attns_v_list).mean(0)   # [l, h, d]
q_mlp_v  = torch.stack(q_mlps_v_list).mean(0)    # [l+1, d]


In [ ]:
## Run the TEXT encoder on the query sentence(s) and average the per-component contributions
prs_t_query = hook_prs_logger_text(model, device, spatial=False)
cast_dtype = model.transformer.get_cast_dtype()
q_attns_t_list, q_mlps_t_list = [], []
with torch.no_grad():
    for sent in query_sentences:
        prs_t_query.reinit()
        text_tokens = tokenizer([sent]).to(device)
        prs_t_query.eos_idx = text_tokens.argmax(dim=-1)
        z0 = model.token_embedding(text_tokens).to(cast_dtype) + model.positional_embedding.to(cast_dtype)
        representation = model.encode_text(text_tokens, normalize=False, attn_method="head_no_spatial")
        prs_t_query.mlps = [prs_t_query._gather_eos(z0).detach().cpu()] + prs_t_query.mlps
        attentions, mlps = prs_t_query.finalize(representation)
        q_attns_t_list.append(attentions[0].float())
        q_mlps_t_list.append(mlps[0].float())
q_attn_t = torch.stack(q_attns_t_list).mean(0)   # [l, h, d]
q_mlp_t  = torch.stack(q_mlps_t_list).mean(0)    # [l+1, d]


In [ ]:
## "Whole imagenet" / "whole text corpus" averages, reusing the already-loaded per-component tensors
ds_attn_v = attns_v.mean(0)   # [l, h, d]
ds_mlp_v  = mlps_v.mean(0)    # [l+1, d]
ds_attn_t = attns_t.mean(0)   # [l, h, d]
ds_mlp_t  = mlps_t.mean(0)    # [l+1, d]


### Head / MLP-layer level heatmaps


In [ ]:
def flatten_components(attn, mlp, kind):
    """
    attn: [l, h, d] mean attention contribution per (layer, head)
    mlp:  [l+1, d]  mean MLP contribution per layer (index 0 = ln_pre/embedding term)
    kind: 'mlp' | 'attn' | 'all' (residual-stream order: mlp[0], attn-layer0-heads, mlp[1], ...)
    Returns (matrix [n_components, d], labels list)
    """
    l, h, d = attn.shape
    if kind == "mlp":
        return mlp, [f"MLP{i}" for i in range(mlp.shape[0])]
    if kind == "attn":
        vecs = attn.reshape(l * h, d)
        labels = [f"L{i}H{j}" for i in range(l) for j in range(h)]
        return vecs, labels
    if kind == "all":
        vecs, labels = [mlp[0]], ["MLP0"]
        for i in range(l):
            for j in range(h):
                vecs.append(attn[i, j]); labels.append(f"L{i}H{j}")
            vecs.append(mlp[i + 1]); labels.append(f"MLP{i + 1}")
        return torch.stack(vecs), labels
    raise ValueError(kind)


def component_heatmap(vecs_a, labels_a, vecs_b, labels_b, metric, title,
                       xlabel="Text-encoder components", ylabel="Vision-encoder components"):
    """vecs_a (rows) x vecs_b (cols) -> dot-product/cosine heatmap."""
    if metric == "cosine":
        vecs_a = vecs_a / vecs_a.norm(dim=-1, keepdim=True)
        vecs_b = vecs_b / vecs_b.norm(dim=-1, keepdim=True)
    sims = (vecs_a @ vecs_b.T).cpu().numpy()
    plt.figure(figsize=(max(6, len(labels_b) * 0.3), max(5, len(labels_a) * 0.3)))
    sns.heatmap(sims, xticklabels=labels_b, yticklabels=labels_a, cmap="RdBu_r", center=0)
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout(); plt.show()
    return sims

In [ ]:
## Head/layer-level heatmaps: query components vs. whole-dataset components, for mlp / attn / all
heatmaps_headlevel = {}
for kind in ["mlp", "attn", "all"]:
    vecs_v_q, labels_v_q = flatten_components(q_attn_v, q_mlp_v, kind)
    vecs_t_q, labels_t_q = flatten_components(q_attn_t, q_mlp_t, kind)
    sims_q = component_heatmap(vecs_v_q, labels_v_q, vecs_t_q, labels_t_q, heatmap_metric,
                                f"[{kind}] query components ({heatmap_metric})")
    heatmaps_headlevel[kind] = (sims_q, labels_v_q, vecs_v_q, labels_t_q, vecs_t_q)

    vecs_v_ds, labels_v_ds = flatten_components(ds_attn_v, ds_mlp_v, kind)
    vecs_t_ds, labels_t_ds = flatten_components(ds_attn_t, ds_mlp_t, kind)
    component_heatmap(vecs_v_ds, labels_v_ds, vecs_t_ds, labels_t_ds, heatmap_metric,
                       f"[{kind}] whole-dataset components ({heatmap_metric})")


### PC-level heatmaps (individual principal-component directions)


In [ ]:
## Vision- and text-head PCs: reuse the PCs already decomposed above (last num_last_layers_ layers)
def head_pcs_to_components(data, max_pcs):
    """One component per (layer, head, princ_comp) with princ_comp < max_pcs; vector = unit vh[princ_comp]."""
    vecs, labels, blocks, heads, pcs = [], [], [], [], []
    for e in data:
        if e["princ_comp"] >= max_pcs:
            continue
        vh = torch.tensor(e["vh"])[e["princ_comp"]]
        vecs.append(vh / vh.norm())
        labels.append(f"L{e['layer']}H{e['head']}P{e['princ_comp']}")
        blocks.append(e["layer"]); heads.append(e["head"]); pcs.append(e["princ_comp"])
    return torch.stack(vecs), labels, blocks, heads, pcs

vecs_v_head_pcs, labels_v_head_pcs, blocks_v_head, heads_v_head, pcs_v_head = head_pcs_to_components(data_v, max_pcs_per_component)
vecs_t_head_pcs, labels_t_head_pcs, blocks_t_head, heads_t_head, pcs_t_head = head_pcs_to_components(data_t, max_pcs_per_component)


In [ ]:
## MLP layers have no precomputed PC decomposition anywhere in this codebase -- run svd_data_approx
## directly on each MLP layer's activation matrix, exactly as compute_text_explanations(.py/_text.py)
## do for heads, restricted to the last num_last_layers_ blocks to match the heads' scope.
def mlp_layer_pcs(mlps, last_layer_idx, tot_layers, max_pcs):
    """
    mlps: [N, L+1, d] per-sample per-MLP-layer contributions.
    Decomposes MLP slots (last_layer_idx+1) .. tot_layers (the MLPs of the last num_last_layers_ blocks).
    Returns (vecs [n,d], labels, blocks) where `block` = the transformer block the MLP belongs to.
    """
    label_bank_np = final_embeddings_texts.cpu().numpy()
    texts_bank = list(texts_str)
    vecs, labels, blocks = [], [], []
    for mlp_slot in range(last_layer_idx + 1, tot_layers + 1):
        block = mlp_slot - 1
        _, info = svd_data_approx(mlps[:, mlp_slot, :].cpu().numpy(), label_bank_np, texts_bank,
                                   block, -1, 5, device, iters=max_pcs)
        vh = torch.tensor(info["vh"])
        for i in range(min(max_pcs, vh.shape[0])):
            vecs.append(vh[i] / vh[i].norm())
            labels.append(f"MLP{mlp_slot}P{i}")
            blocks.append(block)
    return torch.stack(vecs), labels, blocks

vecs_v_mlp_pcs, labels_v_mlp_pcs, blocks_v_mlp = mlp_layer_pcs(mlps_v, last_v, nr_layers_v, max_pcs_per_component)
vecs_t_mlp_pcs, labels_t_mlp_pcs, blocks_t_mlp = mlp_layer_pcs(mlps_t, last_t, nr_layers_t, max_pcs_per_component)


In [ ]:
def pc_components(kind, vecs_head, labels_head, blocks_head, heads_head, vecs_mlp, labels_mlp, blocks_mlp):
    """Assemble the PC-level component list for kind in {'mlp', 'attn', 'all'} (residual-stream order for 'all')."""
    if kind == "mlp":
        return vecs_mlp, labels_mlp
    if kind == "attn":
        return vecs_head, labels_head
    if kind == "all":
        # Order by (block, attn-before-mlp, head, pc): attention heads of a block precede that block's MLP.
        entries = ([(b, 0, h, v, lab) for b, h, v, lab in zip(blocks_head, heads_head, vecs_head, labels_head)]
                   + [(b, 1, -1, v, lab) for b, v, lab in zip(blocks_mlp, vecs_mlp, labels_mlp)])
        entries.sort(key=lambda x: (x[0], x[1], x[2]))
        vecs = torch.stack([e[3] for e in entries])
        labels = [e[4] for e in entries]
        return vecs, labels
    raise ValueError(kind)


In [ ]:
## PC-level heatmaps
heatmaps_pclevel = {}
for kind in ["mlp", "attn", "all"]:
    vecs_v_pc, labels_v_pc = pc_components(kind, vecs_v_head_pcs, labels_v_head_pcs, blocks_v_head, heads_v_head,
                                            vecs_v_mlp_pcs, labels_v_mlp_pcs, blocks_v_mlp)
    vecs_t_pc, labels_t_pc = pc_components(kind, vecs_t_head_pcs, labels_t_head_pcs, blocks_t_head, heads_t_head,
                                            vecs_t_mlp_pcs, labels_t_mlp_pcs, blocks_t_mlp)
    sims_pc = component_heatmap(vecs_v_pc, labels_v_pc, vecs_t_pc, labels_t_pc, heatmap_metric,
                                 f"[{kind}, PC-level] {heatmap_metric}")
    heatmaps_pclevel[kind] = (sims_pc, labels_v_pc, vecs_v_pc, labels_t_pc, vecs_t_pc)


### Most similar (vision, text) component pairs


In [ ]:
def top_matches_for_direction(direction, k=3):
    """Top-k texts (label bank) and top-k image indices (image probes) by cosine to a raw direction."""
    d = direction / direction.norm()
    txt_scores = (final_embeddings_texts / final_embeddings_texts.norm(dim=-1, keepdim=True)) @ d
    img_scores = (final_embeddings_images / final_embeddings_images.norm(dim=-1, keepdim=True)) @ d
    top_txt = torch.topk(txt_scores, k)
    top_img = torch.topk(img_scores, k)
    return [texts_str[i] for i in top_txt.indices.cpu().numpy()], top_img.indices.cpu().numpy()


def print_top_pairs(sims, labels_a, vecs_a, labels_b, vecs_b, k, header):
    print(f"\n=== {header}: top {k} (vision, text) component pairs by |{heatmap_metric}| ===")
    flat_idx = np.argsort(-np.abs(sims), axis=None)[:k]
    for rank, idx in enumerate(flat_idx):
        i, j = np.unravel_index(idx, sims.shape)
        txt_a, img_a = top_matches_for_direction(vecs_a[i])
        txt_b, img_b = top_matches_for_direction(vecs_b[j])
        print(f"#{rank + 1}  vision {labels_a[i]:>12}  <->  text {labels_b[j]:<12}   value={sims[i, j]: .4f}")
        print(f"      vision-side nearest texts: {txt_a}   nearest image idx: {list(img_a)}")
        print(f"      text-side   nearest texts: {txt_b}   nearest image idx: {list(img_b)}")


In [ ]:
for kind, (sims, labels_a, vecs_a, labels_b, vecs_b) in heatmaps_headlevel.items():
    print_top_pairs(sims, labels_a, vecs_a, labels_b, vecs_b, top_k_pairs, header=f"{kind} (head/layer-level, query)")


In [ ]:
for kind, (sims, labels_a, vecs_a, labels_b, vecs_b) in heatmaps_pclevel.items():
    print_top_pairs(sims, labels_a, vecs_a, labels_b, vecs_b, top_k_pairs, header=f"{kind} (PC-level, query)")


In [ ]:
## Intra-modal component similarity: vision components vs. themselves, text components vs. themselves
# Head/MLP-layer level
for kind in ["mlp", "attn", "all"]:
    vecs_v_q, labels_v_q = flatten_components(q_attn_v, q_mlp_v, kind)
    title = f"[{kind}] VISION self-similarity, query ({heatmap_metric})"
    print(title)
    component_heatmap(vecs_v_q, labels_v_q, vecs_v_q, labels_v_q, heatmap_metric, title,
                       xlabel="Vision-encoder components", ylabel="Vision-encoder components")

    vecs_v_ds, labels_v_ds = flatten_components(ds_attn_v, ds_mlp_v, kind)
    title = f"[{kind}] VISION self-similarity, whole-dataset ({heatmap_metric})"
    print(title)
    component_heatmap(vecs_v_ds, labels_v_ds, vecs_v_ds, labels_v_ds, heatmap_metric, title,
                       xlabel="Vision-encoder components", ylabel="Vision-encoder components")

    vecs_t_q, labels_t_q = flatten_components(q_attn_t, q_mlp_t, kind)
    title = f"[{kind}] TEXT self-similarity, query ({heatmap_metric})"
    print(title)
    component_heatmap(vecs_t_q, labels_t_q, vecs_t_q, labels_t_q, heatmap_metric, title,
                       xlabel="Text-encoder components", ylabel="Text-encoder components")

    vecs_t_ds, labels_t_ds = flatten_components(ds_attn_t, ds_mlp_t, kind)
    title = f"[{kind}] TEXT self-similarity, whole-dataset ({heatmap_metric})"
    print(title)
    component_heatmap(vecs_t_ds, labels_t_ds, vecs_t_ds, labels_t_ds, heatmap_metric, title,
                       xlabel="Text-encoder components", ylabel="Text-encoder components")

# PC level (reuses the PCs already built for the cross-modal PC-level heatmaps above)
for kind in ["mlp", "attn", "all"]:
    vecs_v_pc, labels_v_pc = pc_components(kind, vecs_v_head_pcs, labels_v_head_pcs, blocks_v_head, heads_v_head,
                                            vecs_v_mlp_pcs, labels_v_mlp_pcs, blocks_v_mlp)
    title = f"[{kind}, PC-level] VISION self-similarity ({heatmap_metric})"
    print(title)
    component_heatmap(vecs_v_pc, labels_v_pc, vecs_v_pc, labels_v_pc, heatmap_metric, title,
                       xlabel="Vision-encoder components", ylabel="Vision-encoder components")

    vecs_t_pc, labels_t_pc = pc_components(kind, vecs_t_head_pcs, labels_t_head_pcs, blocks_t_head, heads_t_head,
                                            vecs_t_mlp_pcs, labels_t_mlp_pcs, blocks_t_mlp)
    title = f"[{kind}, PC-level] TEXT self-similarity ({heatmap_metric})"
    print(title)
    component_heatmap(vecs_t_pc, labels_t_pc, vecs_t_pc, labels_t_pc, heatmap_metric, title,
                       xlabel="Text-encoder components", ylabel="Text-encoder components")

# Waterbirds: concept removal via the query system (vision, text, and both)
Classic spurious-correlation test: Waterbirds labels (landbird/waterbird) correlate with the
image *background* (land/water) in the training distribution. We use the same query system as
above ("An image of colour black." -> reconstruct via top components) to find the components on
each side that encode a background concept, remove them, and re-evaluate accuracy:
1. **base** -- original images vs. original classifier.
2. **vision-touched** -- background-concept PCs removed from the VISION-encoder reconstruction of
   the images (mirrors `proof_concept_2_remove` in playground.ipynb), vs. original classifier.
3. **text-touched** -- original images vs. a classifier rebuilt with background-concept PCs
   removed from the TEXT-encoder reconstruction of the class-name prompts.
4. **both-touched** -- both sides touched.

The text side has no existing precomputed decomposition (the classifier is normally just 2
template-averaged embeddings, not a dataset), so a couple of cells below build one on the fly:
running the text encoder (with the PRS hook) over every `(class, template)` prompt and decomposing
each head's activations directly with `svd_data_approx`, exactly like the CLI scripts do for a
full dataset. With only 2 classes worth of prompts this basis is necessarily low-data/low-rank --
treat it as illustrative, not as trustworthy as the vision-side decomposition.


In [ ]:
## Waterbirds task parameters
dataset_image_name_wb = "binary_waterbirds"     # assumes compute_activation_values was already
                                                    # run for this dataset (see prepare_data.ipynb)
concepts_to_remove = ["water background", "land background"]
pcs_per_class_start, pcs_per_class_end, pcs_per_class_step = 1, 21, 2
max_pcs_per_head_wb_text = 20   # cap on PCs/head when decomposing the (tiny) text-prompt dataset
text_per_princ_comp_wb = 5      # texts kept per PC when labeling the text-side decomposition


### Vision side: decompose the Waterbirds images (same pipeline as playground.ipynb)


In [ ]:
## Run the chosen algorithm on the VISION-encoder heads for the Waterbirds images
command_wb = (f"python -m utils.scripts.compute_text_explanations "
              f"--device {device} --model {model_name} --algorithm {algorithm} --seed {seed} "
              f"--text_per_princ_comp 20 --num_of_last_layers {num_last_layers_} "
              f"--text_descriptions {dataset_text_name} --dataset {dataset_image_name_wb}")
!{command_wb}


In [ ]:
attention_dataset_wb = f"output_dir/{dataset_image_name_wb}_completeness_{dataset_text_name}_{model_name}_algo_{algorithm}_seed_{seed}.jsonl"

attns_wb = torch.tensor(np.load(f"output_dir/{dataset_image_name_wb}_attn_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device, dtype=torch.float32)  # [N, l, h, d]
mlps_wb  = torch.tensor(np.load(f"output_dir/{dataset_image_name_wb}_mlp_{model_name}_seed_{seed}.npy",  mmap_mode="r")).to(device, dtype=torch.float32)  # [N, l+1, d]
classifier_wb = torch.tensor(np.load(f"output_dir/{dataset_image_name_wb}_classifier_{model_name}.npy", mmap_mode="r")).to(device, dtype=torch.float32)   # [d, C]
labels_wb = torch.tensor(np.load(f"output_dir/{dataset_image_name_wb}_labels_{model_name}_seed_{seed}.npy", mmap_mode="r")).to(device).long()             # [N]

nr_layers_wb = attns_wb.shape[1]
nr_heads_wb  = attns_wb.shape[2]
last_wb = nr_layers_wb - num_last_layers_
classes_wb = waterbird_classes

# Waterbirds background-group metadata (0=land, 1=water background), needed for worst-group accuracy
root_wb = "datasets/waterbird_complete95_forest2water2/"
df_wb = pd.read_csv(root_wb + "metadata.csv")
filtered_df_wb = df_wb[df_wb['split'] == 2]
background_groups_wb = list(filtered_df_wb['place'])

data_wb = get_data(attention_dataset_wb, -1, skip_final=True)
print("WATERBIRDS(vision)  attns", tuple(attns_wb.shape), "mlps", tuple(mlps_wb.shape))


### Text side: decompose the Waterbirds classifier prompts through the text encoder
Runs the text encoder over every `(class, OPENAI_IMAGENET_TEMPLATES)` prompt, capturing per-head/MLP
EOS contributions the same way the on-the-fly query capture does above, then decomposes each of the
last `num_last_layers_` layers' heads with `svd_data_approx` -- the same algorithm the CLI scripts
use, just run directly in the notebook since there is no dataset file for a classifier's own prompts.


In [ ]:
from utils.models.openai_templates import OPENAI_IMAGENET_TEMPLATES

wb_prompts, wb_prompt_labels = [], []
for c, classname in enumerate(classes_wb):
    for template in OPENAI_IMAGENET_TEMPLATES:
        wb_prompts.append(template(classname))
        wb_prompt_labels.append(c)

prs_wb_t = hook_prs_logger_text(model, device, spatial=False)
cast_dtype = model.transformer.get_cast_dtype()
attns_wb_t_list, mlps_wb_t_list = [], []
with torch.no_grad():
    for prompt in wb_prompts:
        prs_wb_t.reinit()
        text_tokens = tokenizer([prompt]).to(device)
        prs_wb_t.eos_idx = text_tokens.argmax(dim=-1)
        z0 = model.token_embedding(text_tokens).to(cast_dtype) + model.positional_embedding.to(cast_dtype)
        representation = model.encode_text(text_tokens, normalize=False, attn_method="head_no_spatial")
        prs_wb_t.mlps = [prs_wb_t._gather_eos(z0).detach().cpu()] + prs_wb_t.mlps
        attentions, mlps = prs_wb_t.finalize(representation)
        attns_wb_t_list.append(attentions[0].float())
        mlps_wb_t_list.append(mlps[0].float())

attns_wb_t = torch.stack(attns_wb_t_list)                       # [T, l, h, d]
mlps_wb_t  = torch.stack(mlps_wb_t_list)                        # [T, l+1, d]
labels_wb_t = torch.tensor(wb_prompt_labels).long()             # [T], class per prompt

nr_layers_wb_t = attns_wb_t.shape[1]
nr_heads_wb_t  = attns_wb_t.shape[2]
last_wb_t = nr_layers_wb_t - num_last_layers_
print("WATERBIRDS(text)  attns", tuple(attns_wb_t.shape), "mlps", tuple(mlps_wb_t.shape),
      f"| {len(classes_wb)} classes x {len(OPENAI_IMAGENET_TEMPLATES)} templates")


In [ ]:
## Decompose each head of the last num_last_layers_ TEXT layers with svd_data_approx (same
## algorithm compute_text_explanations_text.py runs per head), building a get_data()-shaped list
## so reconstruct_embeddings / get_remaining_pcs / reconstruct_all_embeddings_mean_ablation_pcs
## all work unchanged on the text side too.
label_bank_np = final_embeddings_texts.cpu().numpy()
texts_bank = list(texts_str)
data_wb_t = []
for layer in range(last_wb_t, nr_layers_wb_t):
    for head in range(nr_heads_wb_t):
        _, info = svd_data_approx(attns_wb_t[:, layer, head, :].cpu().numpy(), label_bank_np, texts_bank,
                                   layer, head, text_per_princ_comp_wb, device, iters=max_pcs_per_head_wb_text)
        for i, pc in enumerate(info["embeddings_sort"]):
            data_wb_t.append({
                "layer": layer, "head": head, "princ_comp": i,
                "strength_abs": pc["strength_abs"], "strength_rel": pc["strength_rel"],
                "cosine_similarity": pc["cosine_similarity"], "correlation": pc["correlation"],
                "texts": pc["text"], "rank": len(info["embeddings_sort"]),
                "project_matrix": info["project_matrix"], "vh": info["vh"],
                "mean_values_text": info["mean_values_text"], "mean_values_att": info["mean_values_att"],
            })
print(f"Decomposed {len(data_wb_t)} (layer, head, PC) text-side entries for the Waterbirds classifier prompts.")


### Four-way accuracy comparison: base, vision-touched, text-touched, both-touched


In [ ]:
## Rank vision- and text-encoder PCs by correlation with each background concept (query system) --
## computed once, reused for every pcs_per_class value below.
with torch.no_grad():
    concept_embs = torch.stack([
        model.encode_text(tokenizer(c).to(device), normalize=True).to(torch.float32).squeeze(0)
        for c in concepts_to_remove
    ])  # [K, d]
concepts_centered = concept_embs - mean_final_texts  # mean_final_texts from the query section above

sorted_pcs_v, sorted_pcs_t = [], []
for k in range(concepts_centered.shape[0]):
    c = concepts_centered[k:k + 1]
    _, ranked_v = reconstruct_embeddings(data_wb, [c], ["text"], return_princ_comp=True, plot=False,
                                          means=[mean_final_texts], device=device)
    sorted_pcs_v.append(sort_data_by(ranked_v, "correlation_princ_comp_abs", descending=True))
    _, ranked_t = reconstruct_embeddings(data_wb_t, [c], ["text"], return_princ_comp=True, plot=False,
                                          means=[mean_final_texts], device=device)
    sorted_pcs_t.append(sort_data_by(ranked_t, "correlation_princ_comp_abs", descending=True))

## Base accuracy: original images vs. original classifier (doesn't depend on pcs_per_class)
baseline_wb = attns_wb.sum(axis=(1, 2)) + mlps_wb.sum(axis=1)
base_acc, base_idx = test_accuracy(baseline_wb @ classifier_wb, labels_wb, label="Waterbirds baseline")
base_worst = test_waterbird_preds(base_idx, labels_wb, background_groups_wb)


In [ ]:
def unique_entries(entries):
    return list({(e["layer"], e["head"], e["princ_comp"]): e for e in entries}.values())

pcs_range = list(range(pcs_per_class_start, pcs_per_class_end, pcs_per_class_step))
accs_vision, accs_text, accs_both = [], [], []
worst_vision, worst_text, worst_both = [], [], []

for pcs_per_class in pcs_range:
    print(f"--- pcs_per_class = {pcs_per_class} ---")

    # --- Vision side: remove the background-concept PCs from the IMAGE reconstruction ---
    entries_v = unique_entries([e for k in range(len(concepts_to_remove)) for e in top_data(sorted_pcs_v[k], pcs_per_class)])
    remaining_v = get_remaining_pcs(data_wb, entries_v)
    images_touched = reconstruct_all_embeddings_mean_ablation_pcs(
        remaining_v, mlps_wb, attns_wb, attns_wb, nr_layers_wb, nr_heads_wb, last_wb,
        ratio=-1, mean_ablate_all=False).to(device)
    images_touched = images_touched / images_touched.norm(dim=-1, keepdim=True)

    # --- Text side: remove the background-concept PCs from the CLASSIFIER-prompt reconstruction ---
    entries_t = unique_entries([e for k in range(len(concepts_to_remove)) for e in top_data(sorted_pcs_t[k], pcs_per_class)])
    remaining_t = get_remaining_pcs(data_wb_t, entries_t)
    prompts_touched = reconstruct_all_embeddings_mean_ablation_pcs(
        remaining_t, mlps_wb_t, attns_wb_t, attns_wb_t, nr_layers_wb_t, nr_heads_wb_t, last_wb_t,
        ratio=-1, mean_ablate_all=False).to(device)
    prompts_touched = prompts_touched / prompts_touched.norm(dim=-1, keepdim=True)
    classifier_touched = torch.stack([prompts_touched[labels_wb_t == c].mean(0) for c in range(len(classes_wb))])  # [C, d]
    classifier_touched = (classifier_touched / classifier_touched.norm(dim=-1, keepdim=True)).T  # [d, C]

    acc_v, idx_v = test_accuracy(images_touched @ classifier_wb, labels_wb, label=f"vision-touched (n={pcs_per_class})")
    acc_t, idx_t = test_accuracy(baseline_wb @ classifier_touched, labels_wb, label=f"text-touched (n={pcs_per_class})")
    acc_bt, idx_bt = test_accuracy(images_touched @ classifier_touched, labels_wb, label=f"both-touched (n={pcs_per_class})")

    accs_vision.append(acc_v); accs_text.append(acc_t); accs_both.append(acc_bt)
    worst_vision.append(test_waterbird_preds(idx_v, labels_wb, background_groups_wb))
    worst_text.append(test_waterbird_preds(idx_t, labels_wb, background_groups_wb))
    worst_both.append(test_waterbird_preds(idx_bt, labels_wb, background_groups_wb))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].axhline(base_acc, color="gray", linestyle="--", label="base")
axes[0].plot(pcs_range, accs_vision, marker='o', label="vision-touched")
axes[0].plot(pcs_range, accs_text, marker='s', label="text-touched")
axes[0].plot(pcs_range, accs_both, marker='^', label="both-touched")
axes[0].set_xlabel("PCs removed per concept"); axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Overall accuracy"); axes[0].legend(); axes[0].grid(True)

axes[1].axhline(base_worst, color="gray", linestyle="--", label="base")
axes[1].plot(pcs_range, worst_vision, marker='o', label="vision-touched")
axes[1].plot(pcs_range, worst_text, marker='s', label="text-touched")
axes[1].plot(pcs_range, worst_both, marker='^', label="both-touched")
axes[1].set_xlabel("PCs removed per concept"); axes[1].set_ylabel("Worst-group accuracy (%)")
axes[1].set_title("Worst-group accuracy"); axes[1].legend(); axes[1].grid(True)

fig.suptitle(f"Waterbirds background-concept removal ({model_name}): {concepts_to_remove}")
plt.tight_layout()
plt.show()


# Cross-tower bilinear decomposition of the contrastive logit

Both embeddings are exact sums of direct component contributions in the shared space, so the logit decomposes exactly over cross-tower component pairs:

$$\langle M_v, M_t\rangle = \sum_{i\in V}\sum_{j\in T} \langle c_i, d_j\rangle
\qquad\Longrightarrow\qquad
s(I,T) = \sum_{i,j} w_{ij}\,\rho_{ij},\quad
\rho_{ij}=\cos(c_i,d_j),\;
w_{ij}=\frac{\|c_i\|\|d_j\|}{\|M_v\|\|M_t\|}$$

The loss does **not** optimize per-pair cosines — it optimizes the norm-weighted sum with a global normalizer, so a component can escape alignment pressure by shrinking its norm and pairs can cancel; $w$ and $\rho$ are therefore reported separately. All claims below are about **direct** contributions (training gradients also flow through indirect paths), and the final-LN linearization makes the decomposition exact *per input* only.

The cells below (requires the data-loading cells and the `flatten_components`/`component_heatmap` helpers above):

1. **Rank audit** — participation ratio + 90%-variance rank per component from the raw activations (the jsonl `rank` is capped by the algorithm's `iters`, so intrinsic rank must not be read off it).
2. **Centered-cosine head heatmaps** — per-tower mean-centering removes the shared anisotropy direction that pollutes raw dot/cosine heatmaps.
3. **Interaction matrix** $C_{ij} = \mathbb{E}_{pos}\langle c_i, d_j\rangle - \mathbb{E}_{neg}\langle c_i, d_j\rangle = \mathrm{tr}\,\mathrm{Cov}_{pos}(c_i, d_j)$ — the per-pair contribution to the contrastive margin; the means (modality gap / anisotropy) cancel by construction.
4. **Norm-hiding analysis** — $w_{ij}$ vs $\rho_{ij}$: "aligned but muted" vs "loud but uninformative" pairs.
5. **Subspace matching** — principal angles between head PC bases, z-scored against rotation/shuffle nulls + split-half stability; **H1**: pairs with large $|\hat C_{ij}|$ are the pairs whose spectral subspaces align.
6. **Matched pairs** — Hungarian matching with dual-modality text labels, then a **causal coupling test**: paired mean-ablation, predicting sub-additive damage for matched pairs (cross-tower redundancy).

In [ ]:
## Rank audit: participation ratio + 90%-variance rank per head/MLP, from the raw
## activations (the jsonl "rank" = len(embeddings_sort) is capped by the algorithm's
## `iters` argument, so it is NOT an estimate of intrinsic rank)
def spectrum_stats(X):
    # X: [N, d] one component's contributions; eigenvalues of the centered covariance
    Xc = X - X.mean(0, keepdim=True)
    eig = torch.linalg.eigvalsh(Xc.T @ Xc / (X.shape[0] - 1)).clamp(min=0).flip(0)
    tot = eig.sum()
    if tot < 1e-10:  # constant component (e.g. the ln_pre CLS/EOS embedding term)
        return 0.0, 0
    pr = (tot ** 2 / (eig ** 2).sum()).item()             # participation ratio
    r90 = int((eig.cumsum(0) / tot < 0.90).sum().item()) + 1  # 90%-variance rank
    return pr, r90

def tower_rank_audit(attns, mlps, name):
    rows = []
    for l in range(attns.shape[1]):
        for h in range(attns.shape[2]):
            rows.append((f"L{l}H{h}",) + spectrum_stats(attns[:, l, h]))
    for s_ in range(mlps.shape[1]):
        rows.append((f"MLP{s_}",) + spectrum_stats(mlps[:, s_]))
    prs = np.array([r[1] for r in rows]); r90s = np.array([r[2] for r in rows])
    print(f"{name}: participation ratio mean {prs.mean():.1f} (min {prs.min():.1f}, max {prs.max():.1f}) | "
          f"90%-variance rank mean {r90s.mean():.1f} (min {r90s.min()}, max {r90s.max()})")
    return rows

rank_rows_v = tower_rank_audit(attns_v, mlps_v, "VISION")
rank_rows_t = tower_rank_audit(attns_t, mlps_t, "TEXT")
print(f"(for contrast, the capped jsonl 'rank': vision mean {mean_rank_v:.2f}, text mean {mean_rank_t:.2f})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for rows, name in [(rank_rows_v, "vision"), (rank_rows_t, "text")]:
    head_rows = [r for r in rows if r[0].startswith("L")]
    axes[0].hist([r[1] for r in head_rows], bins=30, alpha=0.5, label=name)
    axes[1].hist([r[2] for r in head_rows], bins=30, alpha=0.5, label=name)
axes[0].set_xlabel("participation ratio"); axes[1].set_xlabel("90%-variance rank")
for ax in axes: ax.set_ylabel("# heads"); ax.legend(); ax.grid(True)
fig.suptitle(f"Per-head spectrum statistics ({model_name})")
plt.tight_layout(); plt.show()

In [ ]:
## Dataset-level component heatmaps, cosine metric, per-tower mean-centered:
## subtracting each tower's mean component vector removes the shared anisotropy
## direction that dominates the raw dot/cosine heatmaps above
ds_attn_v_, ds_mlp_v_ = attns_v.mean(0), mlps_v.mean(0)
ds_attn_t_, ds_mlp_t_ = attns_t.mean(0), mlps_t.mean(0)

heatmaps_headlevel_centered = {}
for kind in ["mlp", "attn", "all"]:
    vecs_v_ds, labels_v_ds = flatten_components(ds_attn_v_, ds_mlp_v_, kind)
    vecs_t_ds, labels_t_ds = flatten_components(ds_attn_t_, ds_mlp_t_, kind)
    vecs_v_c = vecs_v_ds - vecs_v_ds.mean(0, keepdim=True)
    vecs_t_c = vecs_t_ds - vecs_t_ds.mean(0, keepdim=True)
    sims_c = component_heatmap(vecs_v_c, labels_v_ds, vecs_t_c, labels_t_ds, "cosine",
                               f"[{kind}] whole-dataset components, per-tower centered (cosine)")
    heatmaps_headlevel_centered[kind] = (sims_c, labels_v_ds, vecs_v_c, labels_t_ds, vecs_t_c)

## Interaction matrix

Over positives $(I, T^+)$ (image ↔ sentence of the same ImageNet class) and independently drawn negatives:

$$C_{ij} \;=\; \mathbb{E}_{pos}\big[\langle c_i(I), d_j(T)\rangle\big] \;-\; \mathbb{E}_{neg}\big[\langle c_i(I), d_j(T)\rangle\big]
\;=\; \mathrm{tr}\,\mathrm{Cov}_{pos}(c_i, d_j)$$

since for independent negatives $\mathbb{E}_{neg}\langle c_i,d_j\rangle = \langle \mathbb{E}c_i, \mathbb{E}d_j\rangle$ — **the means cancel by construction**, removing the modality-gap/anisotropy confound. With class-level pairing and equal samples per class, grand-mean centering over class means makes the cancellation exact. Normalized (RV-coefficient / linear-CKA-like, in $[-1,1]$):

$$\hat C_{ij} = \frac{\mathrm{tr}\,\mathrm{Cov}_{pos}(c_i, d_j)}{\sqrt{\mathrm{tr}\,\mathrm{Cov}(c_i)\cdot\mathrm{tr}\,\mathrm{Cov}(d_j)}}$$

Reported alongside: the norm-weight matrix $W_{ij} = \mathbb{E}\|c_i\|\,\mathbb{E}\|d_j\| / (\mathbb{E}\|M_v\|\,\mathbb{E}\|M_t\|)$ — pairs with high $\hat C$ but tiny $W$ are *aligned but muted*; high $W$ with low $\hat C$ are *loud but uninformative*.

In [ ]:
## Interaction matrix C_ij = tr Cov_pos(c_i, d_j), normalized C_hat, norm-weights W
def flatten_comps(attns, mlps):
    # -> [N, K, d] with K = L*H + (L+1); order: heads (layer-major) then MLP slots
    N, L, H, d = attns.shape
    return torch.cat([attns.reshape(N, L * H, d), mlps], dim=1)

def comp_labels(L, H):
    return [f"L{l}H{h}" for l in range(L) for h in range(H)] + [f"MLP{l}" for l in range(L + 1)]

Xv = flatten_comps(attns_v, mlps_v)          # [N, Kv, d]
Xt = flatten_comps(attns_t, mlps_t)          # [M, Kt, d]
comp_labels_v = comp_labels(nr_layers_v, nr_heads_v)
comp_labels_t = comp_labels(nr_layers_t, nr_heads_t)

C_classes = int(max(labels_v.max(), labels_t.max()).item()) + 1

def class_means(X, labels, C):
    # [C, K, d] per-class mean of each component's contribution
    out = torch.zeros(C, X.shape[1], X.shape[2])
    cnt = torch.zeros(C)
    out.index_add_(0, labels, X)
    cnt.index_add_(0, labels, torch.ones(len(labels)))
    return out / cnt[:, None, None].clamp(min=1)

Mu_v = class_means(Xv, labels_v, C_classes)   # [C, Kv, d]
Mu_t = class_means(Xt, labels_t, C_classes)   # [C, Kt, d]

# Center by grand mean over classes -> the E_neg term cancels exactly
Mu_v_c = Mu_v - Mu_v.mean(0, keepdim=True)
Mu_t_c = Mu_t - Mu_t.mean(0, keepdim=True)

# C_ij = (1/C) sum_c <mu_i^c - mu_i, nu_j^c - nu_j> = trace of between-class cross-covariance
C_mat = torch.einsum('cid,cjd->ij', Mu_v_c, Mu_t_c) / C_classes   # [Kv, Kt]

var_v = torch.einsum('cid,cid->i', Mu_v_c, Mu_v_c) / C_classes
var_t = torch.einsum('cjd,cjd->j', Mu_t_c, Mu_t_c) / C_classes
C_hat = C_mat / (var_v.sqrt()[:, None] * var_t.sqrt()[None, :]).clamp(min=1e-8)

# Norm-weight matrix W_ij = E||c_i|| E||d_j|| / (E||M_v|| E||M_t||)
W_mat = (Xv.norm(dim=-1).mean(0)[:, None] * Xt.norm(dim=-1).mean(0)[None, :]
         / (Xv.sum(1).norm(dim=-1).mean() * Xt.sum(1).norm(dim=-1).mean()))

print(f"interaction matrix: {C_hat.shape[0]} vision x {C_hat.shape[1]} text components, {C_classes} classes")
top = torch.topk(C_hat.abs().flatten(), 15)
print("top |C_hat| pairs:")
for idx, _ in zip(top.indices, top.values):
    i, j = np.unravel_index(idx.item(), tuple(C_hat.shape))
    print(f"  {comp_labels_v[i]:>7} <-> {comp_labels_t[j]:<7}  C_hat={C_hat[i, j].item(): .3f}  "
          f"C={C_mat[i, j].item(): .4f}  W={W_mat[i, j].item():.5f}")

In [ ]:
## C_hat heatmap, per-component marginals, sparsity/participation stats
fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(C_hat.numpy(), cmap="RdBu_r", center=0, ax=ax,
            xticklabels=[l if k % 4 == 0 else "" for k, l in enumerate(comp_labels_t)],
            yticklabels=[l if k % 4 == 0 else "" for k, l in enumerate(comp_labels_v)])
ax.set_xlabel("Text-encoder components"); ax.set_ylabel("Vision-encoder components")
ax.set_title(f"Normalized interaction matrix $\\hat{{C}}$ ({model_name}, class-paired positives)")
plt.tight_layout(); plt.show()

marg_v, marg_t = C_hat.abs().sum(1), C_hat.abs().sum(0)
fig, axes = plt.subplots(2, 1, figsize=(16, 7))
axes[0].bar(range(len(marg_v)), marg_v.numpy())
axes[0].set_xticks(range(0, len(marg_v), 4)); axes[0].set_xticklabels(comp_labels_v[::4], rotation=90, fontsize=6)
axes[0].set_ylabel("sum_j |C_hat|"); axes[0].set_title("Vision components: margin carried")
axes[1].bar(range(len(marg_t)), marg_t.numpy())
axes[1].set_xticks(range(0, len(marg_t), 4)); axes[1].set_xticklabels(comp_labels_t[::4], rotation=90, fontsize=6)
axes[1].set_ylabel("sum_i |C_hat|"); axes[1].set_title("Text components: margin carried")
plt.tight_layout(); plt.show()

# Sparsity: participation ratio of the squared entries + mass share of the top 1% of pairs
c2 = (C_hat ** 2).flatten()
pr_pairs = (c2.sum() ** 2 / (c2 ** 2).sum()).item()
abs_sorted = C_hat.abs().flatten().sort(descending=True).values
top1pct = max(1, len(abs_sorted) // 100)
print(f"pairs: {len(abs_sorted)} | participation ratio of C_hat^2: {pr_pairs:.1f} "
      f"({100 * pr_pairs / len(abs_sorted):.1f}% of pairs) | "
      f"top 1% of pairs carry {100 * abs_sorted[:top1pct].sum().item() / abs_sorted.sum().item():.1f}% of sum|C_hat|")

In [ ]:
## Norm-hiding analysis: rho_ij = E_pos[cos(c_i, d_j)] vs the norm weights W_ij
# pair each image with its class sentence (last sentence of the class if several)
class_to_row = torch.full((C_classes,), -1, dtype=torch.long)
class_to_row[labels_t] = torch.arange(len(labels_t))
pos_rows = class_to_row[labels_v]                     # [N] text row paired with each image
idx_pos = (pos_rows >= 0).nonzero().squeeze(1)

Xv_n = Xv / Xv.norm(dim=-1, keepdim=True).clamp(min=1e-8)
Xt_n = Xt / Xt.norm(dim=-1, keepdim=True).clamp(min=1e-8)
rho_mat = torch.zeros(Xv.shape[1], Xt.shape[1])
for start in range(0, len(idx_pos), 512):
    sel = idx_pos[start:start + 512]
    rho_mat += torch.einsum('nid,njd->ij', Xv_n[sel], Xt_n[pos_rows[sel]])
rho_mat /= len(idx_pos)
del Xv_n, Xt_n

is_head_pair = (torch.tensor([l.startswith("L") for l in comp_labels_v])[:, None]
                & torch.tensor([l.startswith("L") for l in comp_labels_t])[None, :])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for m, sel_mask, lab in [(0, is_head_pair, "head-head"), (1, ~is_head_pair, "involves MLP")]:
    axes[0].scatter(rho_mat[sel_mask], W_mat[sel_mask], s=3, alpha=0.3, label=lab)
    axes[1].scatter(C_hat[sel_mask].abs(), W_mat[sel_mask], s=3, alpha=0.3, label=lab)
axes[0].set_xlabel(r"$\rho_{ij}$ = E$_{pos}$ cos($c_i$, $d_j$)"); axes[1].set_xlabel(r"$|\hat{C}_{ij}|$")
for ax in axes:
    ax.set_ylabel(r"$W_{ij}$ (norm weight)"); ax.set_yscale("log"); ax.legend(); ax.grid(True, alpha=0.3)
fig.suptitle("Norm-hiding: alignment vs the weight it actually gets in the logit")
plt.tight_layout(); plt.show()

# quadrant exemplars
w_med = W_mat.median()
c_hi, c_lo = C_hat.abs().flatten().quantile(0.90), C_hat.abs().flatten().quantile(0.10)
def print_quadrant(mask, header):
    scores = torch.where(mask, C_hat.abs(), torch.zeros_like(C_hat))
    print(header)
    for idx in torch.topk(scores.flatten(), 5).indices:
        i, j = np.unravel_index(idx.item(), tuple(C_hat.shape))
        print(f"  {comp_labels_v[i]:>7} <-> {comp_labels_t[j]:<7}  |C_hat|={C_hat[i, j].abs().item():.3f}  "
              f"rho={rho_mat[i, j].item(): .3f}  W={W_mat[i, j].item():.5f}")
print_quadrant((C_hat.abs() >= c_hi) & (W_mat < w_med), "aligned but muted (high |C_hat|, low W):")
loud = torch.where((C_hat.abs() <= c_lo) & (W_mat >= w_med), W_mat, torch.zeros_like(W_mat))
print("loud but uninformative (high W, low |C_hat|):")
for idx in torch.topk(loud.flatten(), 5).indices:
    i, j = np.unravel_index(idx.item(), tuple(C_hat.shape))
    print(f"  {comp_labels_v[i]:>7} <-> {comp_labels_t[j]:<7}  |C_hat|={C_hat[i, j].abs().item():.3f}  "
          f"rho={rho_mat[i, j].item(): .3f}  W={W_mat[i, j].item():.5f}")

## Cross-tower subspace matching (principal angles)

For vision head $i$ with rank-$k$ PC basis $V_i$ and text head $j$ with $V_j$ (rows orthonormal, from `svd_data_approx`), the principal angles $\theta_1..\theta_k$ come from the SVD of $V_i V_j^\top$ (singular values $=\cos\theta$); similarity = mean $\cos^2\theta$ — sign-invariant, rotation-invariant within the subspace, robust to PC-order instability (chance level $\approx k/d$).

Nulls: **(a)** random orthogonal rotations of the text tower's bases → per-pair z-scores; **(b)** shuffled head assignment (in the H1 cell); **(c)** split-half stability of each head's own basis (an upper bound on how much matching signal the estimation noise allows).

In [ ]:
## Per-head PC bases + principal-angle similarity matrix + nulls
subspace_k = 8

def head_bases(data):
    bases, keys = [], []
    seen = set()
    for e in data:
        key = (e["layer"], e["head"])
        if key in seen or e["princ_comp"] != 0:
            continue
        seen.add(key)
        vh = torch.tensor(e["vh"], dtype=torch.float32)
        if vh.shape[0] < subspace_k:
            continue
        bases.append(vh[:subspace_k])
        keys.append(key)
    return torch.stack(bases), keys          # [n, k, d]

bases_v, heads_dec_v = head_bases(data_v)
bases_t, heads_dec_t = head_bases(data_t)
print(f"{len(heads_dec_v)} vision heads x {len(heads_dec_t)} text heads with >= {subspace_k} PCs "
      f"(chance mean cos^2 ~ {subspace_k / bases_v.shape[-1]:.4f})")

def subspace_sim(Vi, Vj, k=subspace_k):
    # mean cos^2 of the principal angles between the two row-spaces
    s = torch.linalg.svdvals(Vi[:k] @ Vj[:k].T)
    return float((s ** 2).mean())

def subspace_sim_matrix(Bv, Bt):
    M = torch.einsum('ikd,jld->ijkl', Bv, Bt)
    s = torch.linalg.svdvals(M.reshape(-1, subspace_k, subspace_k))
    return (s ** 2).mean(-1).reshape(Bv.shape[0], Bt.shape[0])

S_obs = subspace_sim_matrix(bases_v, bases_t)

# Null (a): random orthogonal rotations of the text-tower bases -> per-pair z-scores
n_rot = 50
gen = torch.Generator().manual_seed(seed)
null_sims = torch.stack([
    subspace_sim_matrix(bases_v, bases_t @ torch.linalg.qr(
        torch.randn(bases_t.shape[-1], bases_t.shape[-1], generator=gen))[0])
    for _ in range(n_rot)])
S_z = (S_obs - null_sims.mean(0)) / null_sims.std(0).clamp(min=1e-8)

# Null (c): split-half stability of each head's own top-k basis (matching-signal ceiling)
def topk_basis(X, k):
    Xc = X - X.mean(0, keepdim=True)
    _, eigvec = torch.linalg.eigh(Xc.T @ Xc)
    return eigvec[:, -k:].flip(1).T           # [k, d]

perm_v = torch.randperm(attns_v.shape[0], generator=gen)
perm_t = torch.randperm(attns_t.shape[0], generator=gen)
stab_v = [subspace_sim(topk_basis(attns_v[perm_v[:len(perm_v) // 2], l, h], subspace_k),
                       topk_basis(attns_v[perm_v[len(perm_v) // 2:], l, h], subspace_k))
          for (l, h) in heads_dec_v]
stab_t = [subspace_sim(topk_basis(attns_t[perm_t[:len(perm_t) // 2], l, h], subspace_k),
                       topk_basis(attns_t[perm_t[len(perm_t) // 2:], l, h], subspace_k))
          for (l, h) in heads_dec_t]
print(f"split-half basis stability (mean cos^2): vision {np.mean(stab_v):.3f}, text {np.mean(stab_t):.3f}")
print(f"observed cross-tower similarity: mean {S_obs.mean().item():.4f}, max {S_obs.max().item():.4f} "
      f"(rotation-null mean {null_sims.mean().item():.4f})")

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(S_z.numpy(), cmap="RdBu_r", center=0, ax=ax,
            xticklabels=[f"L{l}H{h}" for (l, h) in heads_dec_t],
            yticklabels=[f"L{l}H{h}" for (l, h) in heads_dec_v])
ax.set_xlabel("Text heads"); ax.set_ylabel("Vision heads")
ax.set_title(f"Subspace-alignment z-scores vs rotation null (k={subspace_k}, {n_rot} rotations)")
plt.tight_layout(); plt.show()

In [ ]:
## H1 (headline test): does |C_hat| predict cross-tower subspace alignment?
from scipy.stats import spearmanr

rows_h1 = [comp_labels_v.index(f"L{l}H{h}") for (l, h) in heads_dec_v]
cols_h1 = [comp_labels_t.index(f"L{l}H{h}") for (l, h) in heads_dec_t]
x_h1 = C_hat[rows_h1][:, cols_h1].abs().flatten().numpy()
y_h1 = S_obs.flatten().numpy()

rho_h1, p_h1 = spearmanr(x_h1, y_h1)

# Null (b): shuffled head assignment
n_perm = 1000
rng = np.random.RandomState(seed)
null_rho = np.array([spearmanr(x_h1, rng.permutation(y_h1))[0] for _ in range(n_perm)])
z_h1 = (rho_h1 - null_rho.mean()) / null_rho.std()
print(f"H1: spearman(|C_hat|, subspace similarity) over {len(x_h1)} head pairs = {rho_h1:.3f} "
      f"(p={p_h1:.2e}, z={z_h1:.1f} vs {n_perm} shuffled assignments)")
if rho_h1 > 0 and z_h1 > 3:
    print("-> alignment is routed through the pairs that carry the contrastive margin")
else:
    print("-> weak/no relation: norm-weighting / cancellation dominates over subspace alignment")

plt.figure(figsize=(7, 5))
plt.scatter(x_h1, y_h1, s=8, alpha=0.4)
plt.xscale("log")
plt.xlabel(r"$|\hat{C}_{ij}|$ (contribution to the contrastive margin)")
plt.ylabel(f"subspace similarity (mean cos$^2$, k={subspace_k})")
plt.title(f"H1: spearman={rho_h1:.3f}, z={z_h1:.1f} ({model_name})")
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
## Bipartite matching + qualitative table (matched pairs with their PC-0 text labels)
from scipy.optimize import linear_sum_assignment

ri, ci = linear_sum_assignment(-S_z.numpy())          # one-to-one, maximize total z
order_match = np.argsort(-S_z.numpy()[ri, ci])

def pc0_texts(data, layer, head, n=3):
    for e in data:
        if e["layer"] == layer and e["head"] == head and e["princ_comp"] == 0:
            return [v for d in e["texts"] for k, v in d.items() if k.startswith("text")][:n]
    return []

print("=== Hungarian matching, top pairs by z-score ===")
for k in order_match[:15]:
    (lv, hv), (lt, ht) = heads_dec_v[ri[k]], heads_dec_t[ci[k]]
    i, j = comp_labels_v.index(f"L{lv}H{hv}"), comp_labels_t.index(f"L{lt}H{ht}")
    print(f"vision L{lv}H{hv} <-> text L{lt}H{ht}: sim={S_obs[ri[k], ci[k]].item():.3f} "
          f"z={S_z[ri[k], ci[k]].item():.1f} |C_hat|={C_hat[i, j].abs().item():.3f}")
    print(f"   vision PC0: {pc0_texts(data_v, lv, hv)}")
    print(f"   text   PC0: {pc0_texts(data_t, lt, ht)}")

# many-to-one view: best vision head for each text head
best_v = S_z.argmax(0)
print("\n=== per-text-head argmax vision head (many-to-one) ===")
for j, (lt, ht) in enumerate(heads_dec_t):
    lv, hv = heads_dec_v[best_v[j]]
    print(f"text L{lt}H{ht} -> vision L{lv}H{hv}  (z={S_z[best_v[j], j].item():.1f})")

In [ ]:
## Causal coupling: paired mean-ablation of matched (vision head, text head) pairs.
## Prediction: sub-additive damage (interaction < 0) for matched pairs, i.e. cross-tower
## redundancy, relative to importance-matched control pairs.
sum_v_full = attns_v.sum(dim=(1, 2)) + mlps_v.sum(1)
sum_t_full = attns_t.sum(dim=(1, 2)) + mlps_t.sum(1)
mean_attn_v = attns_v.mean(0)
mean_attn_t = attns_t.mean(0)

@torch.no_grad()
def joint_zero_shot_acc(img_emb, txt_emb):
    # classifier = per-class prototypes of the (possibly ablated) text embeddings
    W = torch.zeros(C_classes, txt_emb.shape[1])
    cnt = torch.zeros(C_classes)
    W.index_add_(0, labels_t, txt_emb)
    cnt.index_add_(0, labels_t, torch.ones(len(labels_t)))
    W = W / cnt[:, None].clamp(min=1)
    W = W / W.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    imgs = img_emb / img_emb.norm(dim=-1, keepdim=True)
    return ((imgs @ W.T).argmax(1) == labels_v).float().mean().item() * 100

def embeds_ablated(base_sum, attns, mean_attn, heads):
    e = base_sum.clone()
    for l, h in heads:
        e += mean_attn[l, h] - attns[:, l, h]
    return e

base_acc_joint = joint_zero_shot_acc(sum_v_full, sum_t_full)
print(f"base joint zero-shot accuracy: {base_acc_joint:.2f}%")

damage_v = {hd: base_acc_joint - joint_zero_shot_acc(
                embeds_ablated(sum_v_full, attns_v, mean_attn_v, [hd]), sum_t_full)
            for hd in heads_dec_v}
damage_t = {hd: base_acc_joint - joint_zero_shot_acc(
                sum_v_full, embeds_ablated(sum_t_full, attns_t, mean_attn_t, [hd]))
            for hd in heads_dec_t}

n_causal = 5
matched_pairs = [(heads_dec_v[ri[k]], heads_dec_t[ci[k]]) for k in order_match[:n_causal]]
matched_t_set = {b for _, b in matched_pairs}

inter_matched, inter_control, pair_names = [], [], []
for A, B in matched_pairs:
    # control text head: closest solo damage to B among non-matched heads
    ctrl = min((hd for hd in heads_dec_t if hd not in matched_t_set),
               key=lambda hd: abs(damage_t[hd] - damage_t[B]))
    img_A = embeds_ablated(sum_v_full, attns_v, mean_attn_v, [A])
    dmg_AB = base_acc_joint - joint_zero_shot_acc(
        img_A, embeds_ablated(sum_t_full, attns_t, mean_attn_t, [B]))
    dmg_ACtrl = base_acc_joint - joint_zero_shot_acc(
        img_A, embeds_ablated(sum_t_full, attns_t, mean_attn_t, [ctrl]))
    inter_matched.append(dmg_AB - damage_v[A] - damage_t[B])
    inter_control.append(dmg_ACtrl - damage_v[A] - damage_t[ctrl])
    pair_names.append(f"L{A[0]}H{A[1]}\n+L{B[0]}H{B[1]}")
    print(f"vision L{A[0]}H{A[1]} + text L{B[0]}H{B[1]}: dmg_v={damage_v[A]:.2f} dmg_t={damage_t[B]:.2f} "
          f"joint={dmg_AB:.2f} interaction={inter_matched[-1]:+.2f} | "
          f"control text L{ctrl[0]}H{ctrl[1]} (dmg={damage_t[ctrl]:.2f}) interaction={inter_control[-1]:+.2f}")

print(f"\nmean interaction: matched {np.mean(inter_matched):+.3f} vs control {np.mean(inter_control):+.3f} "
      f"(cross-tower redundancy => matched more negative)")

x = np.arange(n_causal)
plt.figure(figsize=(9, 4.5))
plt.bar(x - 0.2, inter_matched, 0.4, label="matched pair")
plt.bar(x + 0.2, inter_control, 0.4, label="importance-matched control")
plt.axhline(0, color="gray", lw=0.8)
plt.xticks(x, pair_names, fontsize=8)
plt.ylabel("interaction = dmg(A+B) - dmg(A) - dmg(B)  [acc %]")
plt.title(f"Paired-ablation interaction ({model_name}): sub-additivity test")
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()